# A/B de dos palancas de vLLM en nuestra carga real (Colab, **L4**)

Palancas sacadas de `sybil-solutions/inference-optimisation-skill`, medidas en **nuestra** carga: Qwen3.5-0.8B, 150 problemas GSM8K con thinking, nuestro muestreo, el corte temprano de loops y las mismas opciones de vLLM que `discover_math500_blind._llm_fast`. Cada corrida es un proceso nuevo, igual que cada paso de los notebooks.

**1. Caches de compilación entre sesiones** (solo arranque del motor):
- `cold`: todo vacío · `warm_local` ×2: todo caliente en disco local
- `drive_cold` / `drive_now`: `VLLM_CACHE_ROOT` en Drive como hoy; Inductor/Triton/FlashInfer se pierden con la VM → **lo que pagan hoy los notebooks en cada sesión nueva**
- `drive_full`: un solo `.tar` con todas las caches en Drive, descomprimido a disco local (se mide también el tiempo de copia)

**2. KV en fp8 vs auto** (generación completa, intercalado auto/fp8/auto/fp8/auto/fp8/auto): `AGENTS.md` lo prende en L4, pero el Qwen3.5 tiene KV en solo 1 de cada 4 capas y el skill reporta fp8 KV *más lento* en un kernel. fp8 gana solo si supera a **los dos** controles vecinos. También: precisión, tasa de loops, tamaño de KV y "Maximum concurrency" (¿caben de verdad las 150 secuencias?).

**Uso:** Runtime → **L4** → Run all. **~45–60 min.** Reanudable: el estado va a `MyDrive/wsi_runs/_bench_vllm_levers`. Resultado: `MyDrive/wsi_runs/FINALES/bench_vllm_levers.json`. Al final la máquina se libera sola.


In [ ]:
RUN_NAME = "bench_vllm_levers"
MODEL = "Qwen/Qwen3.5-0.8B"
REPO = "/content/wsi"
BASE = "/content/base"
W = "/content/work"
S = f"{REPO}/scripts/bench_vllm_levers.py"


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q vllm hf_transfer
!pip install -q flash-linear-attention || echo "fla opcional"
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)


In [ ]:
import os, glob, shutil, json, subprocess, datetime, time
from google.colab import drive
drive.mount("/content/drive")
DRIVE = f"/content/drive/MyDrive/wsi_runs/_{RUN_NAME}"     # estado + caches de prueba (no toca _vllm_cache real)
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
os.makedirs(DRIVE, exist_ok=True); os.makedirs(FINALES, exist_ok=True); os.makedirs(W, exist_ok=True)
if os.path.exists(f"{DRIVE}/plan_state.json") and not os.path.exists(f"{W}/plan_state.json"):
    shutil.copy2(f"{DRIVE}/plan_state.json", W)            # reanudar
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.pop("VLLM_CACHE_ROOT", None)                     # el script controla las caches de cada corrida


In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
assert os.path.exists(S), "bench_vllm_levers.py no está en GitHub todavía: falta hacer push"
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)


In [ ]:
# un vLLM que quedó de una celda interrumpida tiene la GPU ocupada: matarlo primero
!nvidia-smi --query-compute-apps=pid --format=csv,noheader | xargs -r kill -9
def run_step(cmd):
    print("
>>>", " ".join(cmd), flush=True)
    p = subprocess.run(cmd, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Falló: {' '.join(cmd)}")

if not os.path.exists(f"{W}/requests_test.jsonl"):
    run_step(["python", "-u", S, "prepare", "--work", W, "--base-dir", BASE, "--n", "150", "--max-new", "4096"])
run_step(["python", "-u", S, "plan", "--work", W, "--drive-dir", DRIVE, "--local-root", "/content/levers_cache", "--kv-pairs", "3"])
run_step(["python", "-u", S, "report", "--work", W])


In [ ]:
# resultado final en Drive (solo si está completo)
src = f"{W}/results_bench_vllm_levers.json"
state = json.load(open(f"{W}/plan_state.json"))
assert state.get("complete") and os.path.exists(src), "la corrida no terminó; revisa el error arriba"
res = json.load(open(src))
res["runs"] = {k: {kk: vv for kk, vv in v.items() if kk not in ("correct_ids", "log_tail")} for k, v in state["runs"].items()}
res["_final"] = {"complete": True, "model": MODEL, "commit": COMMIT, "notebook": "bench_vllm_levers_colab.ipynb",
                 "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
                 "vllm": vllm.__version__,
                 "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as f:
    json.dump(res, f, indent=1)
os.replace(dst + ".tmp", dst)
print("Guardado final en Drive:", dst)
print(json.dumps({"startup": res["startup"], "kv": {k: v for k, v in res["kv"].items()}}, indent=1))


In [ ]:
# apagado automático: solo si el resultado completo ya está en FINALES
AUTO_APAGAR = True
dst = f"{FINALES}/{RUN_NAME}.json"
if AUTO_APAGAR and os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete"):
    print("Resultado en FINALES. Liberando la máquina en 90 s...")
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No se apaga: falta el resultado completo en FINALES.")
